# GloVe (mean pooling, concatenated) + unidirectional GRU + MLP

**Task:** Abductive Commonsense Reasoning (ART). Given two observations O1, O2 and two
hypotheses H1, H2, pick the hypothesis that better explains what happened in between.

**Approach of this notebook:** Same as `rnn_basic.ipynb`, only the recurrent layer changes:

- same preprocessing and GloVe sentence vectors (mean pooling, `glove-wiki-gigaword-300`)
- the concatenated 1200 dim vector is read as a sequence of 4 time steps O1 -> O2 -> H1 -> H2, 300 dims each
- one layer, unidirectional GRU (`nn.GRU`), its last hidden state summarizes the whole example
- MLP head: linear -> ReLU -> linear -> 2 logits
- softmax over the 2 logits -> probability for H1 and H2, the answer is the more probable hypothesis

In [1]:
%pip install -q datasets gensim lightning nltk python-dotenv wandb


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))  # notebooks/project_1 -> shared pipeline package

import lightning as L
import torch
import torch.nn as nn

from pipeline import tracking
from pipeline.data import DATASET, load_splits
from pipeline.evaluation import predict_hypotheses
from pipeline.features.static_embeddings import MeanEmbedder, build_features, combine_concat, load_vectors
from pipeline.loaders import make_loader
from pipeline.preprocessing import tokenize_split
from pipeline.preprocessing import nltk_alnum_lower as preprocessing  # swap the preprocessing variant here
from pipeline.training import evaluate_best, fit, training_summary, weights_dir

True

In [3]:
tracking.login()

environment loaded from: /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/.env


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: cmedi32 (cmedi32-hochschule-luzern) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
RUN_NAME = "glove-concatenate-gru-unidirectional"  # W&B run + artifact name, weights in models/project_1/glove_concatenate_gru_unidirectional/weights

config = {
    "dataset": DATASET,
    **preprocessing.CONFIG,  # tokenizer, lowercase, remove_punctuation, remove_stopwords, normalization, truncation
    "embedding": "glove-wiki-gigaword-300",
    "sentence_representation": "mean",
    "example_representation": "sequence",  # O1, O2, H1, H2 -> 4 rnn time steps
    "encoder": "gru",  # rnn, gru, lstm
    "bidirectional": False,
    "num_layers": 1,
    "rnn_hidden_dim": 256,
    "classifier": "mlp",  # linear -> relu -> linear
    "hidden_dim": 256,
    "learning_rate": 1e-3,
    "batch_size": 64,
    "max_epochs": 100,
    "patience": 5,  # validation checks without improvement of loss/valid before stopping
    "seed": 42,
}

run = tracking.init_run(RUN_NAME, config)

In [5]:
train, valid, test = load_splits()

train: 16801, valid: 1000, test: 1532


In [8]:
train_tokens = tokenize_split(train, preprocessing.preprocess)
valid_tokens = tokenize_split(valid, preprocessing.preprocess)
test_tokens = tokenize_split(test, preprocessing.preprocess)

In [9]:
wv = load_vectors(config["embedding"])

vocabulary: 400000 words, 300 dimensions


In [12]:
embed = MeanEmbedder(wv)

X_train, y_train = build_features(train_tokens, embed, combine_concat) # this simple change in combination with much lower OOV rate made a huge difference.
X_valid, y_valid = build_features(valid_tokens, embed, combine_concat)
X_test, y_test = build_features(test_tokens, embed, combine_concat)

### DataLoaders - Takes care of Training Batching and Loading

In [13]:
L.seed_everything(config["seed"])

train_loader = make_loader(X_train, y_train, config["batch_size"], shuffle=True)
valid_loader = make_loader(X_valid, y_valid, config["batch_size"], shuffle=False)
test_loader = make_loader(X_test, y_test, config["batch_size"], shuffle=False)

Seed set to 42


### LightningModule Model

In [14]:
class GRUClassifier(L.LightningModule):

    def __init__(self, input_dim, rnn_hidden_dim, hidden_dim, output_dim, num_layers, bidirectional, learning_rate):
        super().__init__()
        self.save_hyperparameters()
        self.rnn = nn.GRU(input_dim, rnn_hidden_dim, num_layers=num_layers, bidirectional=bidirectional, batch_first=True)
        num_directions = 2 if bidirectional else 1
        self.classifier = nn.Sequential(
            nn.Linear(rnn_hidden_dim * num_directions, hidden_dim),  # bidirectional: forward and backward state concatenated
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, x):
        # (batch, 4 * 300) -> (batch, 4, 300): one time step per sentence O1, O2, H1, H2
        sequence = x.view(x.size(0), -1, self.hparams.input_dim)
        _, hidden = self.rnn(sequence)  # hidden: (num_layers * num_directions, batch, rnn_hidden_dim)
        if self.hparams.bidirectional:
            features = torch.cat([hidden[-2], hidden[-1]], dim=1)  # last layer: forward state after H2, backward state after O1
        else:
            features = hidden[-1]  # last layer: state after reading H2
        return self.classifier(features)  # logits: (batch, 2)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)

    def shared_step(self, batch, split):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)  # CrossEntropyLoss applies the softmax itself, so it gets the raw logits
        accuracy = (logits.argmax(dim=1) == y).float().mean()
        self.log(f"loss/{split}", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log(f"accuracy/{split}", accuracy, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self.shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self.shared_step(batch, "valid")

    def test_step(self, batch, batch_idx):
        return self.shared_step(batch, "test")

    def predict_step(self, batch, batch_idx):
        x, _ = batch
        return torch.softmax(self(x), dim=1)  # (batch, 2): p(hypothesis_1), p(hypothesis_2), each row sums to 1

In [15]:
model = GRUClassifier(
    input_dim=wv.vector_size,  # one time step = one 300 dim sentence vector
    rnn_hidden_dim=config["rnn_hidden_dim"],
    hidden_dim=config["hidden_dim"],
    output_dim=2,
    num_layers=config["num_layers"],
    bidirectional=config["bidirectional"],
    learning_rate=config["learning_rate"],
)

trainer = fit(model, train_loader, valid_loader, weights_dir(RUN_NAME), max_epochs=config["max_epochs"], patience=config["patience"])

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/loggers/wandb.py:400: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ rnn        │ GRU              │  428 K │ train │     0 │
│ 1 │ classifier │ Sequential       │ 66.3 K │ train │     0 │
│ 2 │ loss_fn    │ CrossEntropyLoss │      0 │ train │     0 │
└───┴────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 494 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 494 K                                                                                                
Total estimated model params size (MB): 1.979                                                                      
Modules in train mode: 6                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, 
LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.

Metric loss/valid improved. New best score: 0.651
Metric loss/valid improved by 0.011 >= min_delta = 0.0. New best score: 0.640
Metric loss/valid improved by 0.009 >= min_delta = 0.0. New best score: 0.631
Monitored metric loss/valid did not improve in the last 5 records. Best score: 0.631. Signaling Trainer to stop.


### Best Checkpoint + Upload Artifacts to Weights and Biases

In [16]:
evaluate_best(trainer, model, valid_loader, test_loader)

run.summary.update({
    **training_summary(trainer, y_test),  # accuracy/majority, best/loss_valid, best/checkpoint, early_stopping/stopped_epoch
    **embed.summary(),  # oov/tokens, oov/occurrences, oov/rate, oov/unique_types, oov/empty_sentences
})

Restoring states from the checkpoint path at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_concatenate_gru_unidirectional/weights/best-epoch=3.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_concatenate_gru_unidirectional/weights/best-epoch=3.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      accuracy/valid       │    0.6430000066757202     │
│        loss/valid         │    0.6314162611961365     │
└───────────────────────────┴───────────────────────────┘

Restoring states from the checkpoint path at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_concatenate_gru_unidirectional/weights/best-epoch=3.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_concatenate_gru_unidirectional/weights/best-epoch=3.ckpt


Output()

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       accuracy/test       │    0.5097911357879639     │
│         loss/test         │    0.7967742085456848     │
└───────────────────────────┴───────────────────────────┘

In [17]:
tracking.log_best_checkpoint(run, RUN_NAME, trainer.checkpoint_callback)

<Artifact glove-concatenate-gru-unidirectional>

### Probability per Hypothesis and Answer

In [18]:
results = predict_hypotheses(trainer, model, test_loader, test)
results.head(10)

Restoring states from the checkpoint path at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_concatenate_gru_unidirectional/weights/best-epoch=3.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_concatenate_gru_unidirectional/weights/best-epoch=3.ckpt


Output()

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


accuracy from the answers: 0.5098


,observation_1,observation_2,hypothesis_1,hypothesis_2,label,p_hypothesis_1,p_hypothesis_2,answer
0,A man drove while listening to the radio.,The man called the radio station to ask.,The man heard a song he liked but the radio st...,He wanted to know the name of a song.,2,0.372067,0.627933,2
1,Ada was emptying the trash one night.,Ada had to pick up all the trash from the floor.,the bag broke as Ada was walking out the door.,The trash bag ripped open from a hole in the top.,1,0.598008,0.401992,1
2,Alex was home alone on a Friday night.,"Three friends were there to visit him, and the...",Alex was sad to spend the whole evening alone.,"Alex,posted on facebook that he was lonely at ...",2,0.297325,0.702675,2
3,Amber always thought she was the best tennis p...,Amber was disappointed that she lost to a new ...,amber is tennis player so good for playing but...,She be saded against a new player.,1,0.214147,0.785853,2
4,Amy decided to move from Wisconsin to Florida.,"However, she discovered Alaska was a lot of fun.",Amy did not think Alaska would be and decided ...,Amy did not think Alaska would be fun but deci...,2,0.526565,0.473435,1
5,Ann loved cake.,Ann came inside and watched as cakes were bein...,The local baker let Ann come into the workshop...,She passed by a meat shop one day.,1,0.794463,0.205537,1
6,Anna was jogging with her phone in her pocket.,She was so relieved to see it wasn't cracked!,Anne dropped her phone.,Anna's phone fell out of her pocket into nearb...,1,0.650377,0.349623,1
7,Beth was a model in a fashion show at her school.,"Beth missed the show, and was unable to tell a...",Beth passed out from overeating.,Beth went out and broke her phone with no ride...,2,0.558863,0.441137,1
8,Beth's brother Jake came to her house to check...,Jake slept on the couch to keep her safe.,Jake couldn't find the source of the noise so ...,Jake was scared to go to sleep alone in the ho...,1,0.458234,0.541766,2
9,Bob decided to hike in the jungles of Indonesia.,Bob had also forgotten his shoes.,Bob stopped in the middle of the hike because ...,Bob stopped in the middle of his hike to tie h...,1,0.461988,0.538012,2


In [19]:
run.finish()

accuracy/test,▁
accuracy/train,▁▄▄▄▅▆▆▇█
accuracy/valid,▂▃▁▅▃▄▄█▇▅
epoch,▁▁▂▂▃▃▃▃▄▄▅▅▆▆▆▆▇▇██
loss/test,▁
loss/train,█▇▆▆▅▅▄▂▁
loss/valid,▃▂▂▁▂▂▅▅█▁
trainer/global_step,▁▁▂▂▃▃▄▄▄▄▅▅▆▆▇▇████
accuracy/majority,0.50979
accuracy/test,0.50979
accuracy/train,0.77847
